In [10]:
!pip install PySastrawi


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# =========================================================
# IMPORT LIBRARY
# =========================================================
import sys
# =========================================================
# IMPORT LIBRARY
# =========================================================
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
from supabase import create_client, Client
SUPABASE_URL = 'https://bnuzmrtiaciqlotxcgot.supabase.co'
SUPABASE_KEY = 'sb_publishable_Z8M8GISPVKMp1SGrQHrlLg_AZa8EOo-'
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

from src.preprocessing.clean_text import clean_text
from src.preprocessing.tokenizing import tokenizing
from src.preprocessing.stopwords_id import get_stopwords
from src.preprocessing.stemming import stemming

print("✅ Import berhasil")



✅ Import berhasil


In [12]:
# =========================================================
# LOAD DATA DARI SUPABASE (AUTO BATCH)
# =========================================================
print("📥 Mengambil data dari Supabase...")

all_data = []
batch_size = 1000
offset = 0

while True:
    response = (
        supabase.table("scholar_articles")
        .select("id, title, abstract, authors, year, source, category, pdf_url, url, scrape_status")
        .range(offset, offset + batch_size - 1)
        .execute()
    )

    batch = response.data
    if not batch:
        break

    all_data.extend(batch)
    print(f"  → Batch {offset // batch_size + 1}: {len(batch)} data")

    if len(batch) < batch_size:
        break
    offset += batch_size

df = pd.DataFrame(all_data)
print(f"✅ Total data: {len(df)} baris")
df.head(3)

📥 Mengambil data dari Supabase...
  → Batch 1: 200 data
✅ Total data: 200 baris


,id,title,abstract,authors,year,source,category,pdf_url,url,scrape_status
0,1,Machine learning for microbiologists,… how to evaluate a machine learning model and...,"F Asnicar, AM Thomas, A Passerini…",2024,Nature Reviews …,machine learning,https://pmc.ncbi.nlm.nih.gov/articles/PMC11980...,https://www.nature.com/articles/s41579-023-009...,pdf_failed
1,2,International conference on machine learning,"In this paper, we make the key delineation on ...","W Li, C Wang, G Cheng, Q Song",2023,Transactions on machine learning …,machine learning,https://par.nsf.gov/servlets/purl/10418406,https://par.nsf.gov/servlets/purl/10418406,pdf_failed
2,3,What is machine learning?,… that one can employ in machine learning (ML)...,J Bell,2022,Machine learning and the city: applications in …,machine learning,None,https://onlinelibrary.wiley.com/doi/abs/10.100...,no_pdf


In [13]:
# =========================================================
# VALIDASI KOLOM WAJIB
# =========================================================
required_cols = ["id", "title", "abstract", "authors", "year", "source", "category", "pdf_url", "url", "scrape_status"]
missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_cols}")

# Isi null minimal untuk field teks utama
df["title"] = df["title"].fillna("")
df["abstract"] = df["abstract"].fillna("")

print("✅ Kolom wajib valid")

✅ Kolom wajib valid


In [14]:
# =========================================================
# STEP 1 - TEXT CLEANSING
# =========================================================
# Gabungkan title + abstract dulu
df["full_text"] = (df["title"] + " " + df["abstract"]).str.strip()

# Cleansing (lowercase sudah dilakukan di clean_text)
df["cleaned"] = df["full_text"].apply(clean_text)

print("✅ Step 1 selesai: Text Cleansing")
df[["full_text", "cleaned"]].head(3)

✅ Step 1 selesai: Text Cleansing


,full_text,cleaned
0,Machine learning for microbiologists … how to ...,Machine learning for microbiologists how to ev...
1,International conference on machine learning I...,International conference on machine learning I...
2,What is machine learning? … that one can emplo...,What is machine learning that one can employ i...


In [15]:
# =========================================================
# STEP 2 - TOKENIZATION
# =========================================================
df["tokens"] = df["cleaned"].apply(tokenizing)

print("✅ Step 2 selesai: Tokenization")
df[["cleaned", "tokens"]].head(3)

✅ Step 2 selesai: Tokenization


,cleaned,tokens
0,Machine learning for microbiologists how to ev...,"[machine, learning, for, microbiologists, how,..."
1,International conference on machine learning I...,"[international, conference, on, machine, learn..."
2,What is machine learning that one can employ i...,"[what, is, machine, learning, that, one, can, ..."


In [16]:
# =========================================================
# STEP 3 - STOPWORD REMOVAL
# =========================================================
stop_words = get_stopwords()

def remove_stopwords(tokens):
    if not isinstance(tokens, list):
        return []
    return [t for t in tokens if t not in stop_words and len(t) > 1]

df["tokens_clean"] = df["tokens"].apply(remove_stopwords)

print("✅ Step 3 selesai: Stopword Removal")
df[["tokens", "tokens_clean"]].head(3)

✅ Step 3 selesai: Stopword Removal


,tokens,tokens_clean
0,"[machine, learning, for, microbiologists, how,...","[machine, learning, for, microbiologists, how,..."
1,"[international, conference, on, machine, learn...","[international, conference, on, machine, learn..."
2,"[what, is, machine, learning, that, one, can, ...","[what, is, machine, learning, that, one, can, ..."


In [17]:
# =========================================================
# STEP 4 - STEMMING (Sastrawi / Nazief-Adriani)
# =========================================================
df["tokens_stemmed"] = df["tokens_clean"].apply(stemming)

print("✅ Step 4 selesai: Stemming")
df[["tokens_clean", "tokens_stemmed"]].head(3)

✅ Step 4 selesai: Stemming


,tokens_clean,tokens_stemmed
0,"[machine, learning, for, microbiologists, how,...","[machine, learning, for, microbiologists, how,..."
1,"[international, conference, on, machine, learn...","[international, conference, on, machine, learn..."
2,"[what, is, machine, learning, that, one, can, ...","[what, is, machine, learning, that, one, can, ..."


In [18]:
# =========================================================
# GABUNGKAN TOKEN → CLEANED_TEXT
# =========================================================
df['cleaned_text'] = df['tokens_stemmed'].apply(
    lambda x: ' '.join(x) if isinstance(x, list) else '')

# Baru cek di sini ✅
print(f'Total baris diproses: {len(df)}')
print(f"📊 cleaned_text kosong: {int(df['cleaned_text'].eq('').sum())}")
df[["title", "cleaned_text"]].head(5)

Total baris diproses: 200
📊 cleaned_text kosong: 0


,title,cleaned_text
0,Machine learning for microbiologists,machine learning for microbiologists how to ev...
1,International conference on machine learning,international conference on machine learning i...
2,What is machine learning?,what is machine learning that one can employ i...
3,Amnesiac machine learning,amnesiac machine learning it gives eu resident...
4,Designing nanotheranostics with machine learning,designing nanotheranostics with machine learni...


In [19]:
# =========================================================
# AUDIT KUALITAS PREPROCESSING
# =========================================================
df["len_full_text"] = df["full_text"].apply(lambda x: len(str(x).split()))
df["len_cleaned_text"] = df["cleaned_text"].apply(lambda x: len(str(x).split()))

print("Rata-rata token sebelum preprocessing:", round(df["len_full_text"].mean(), 2))
print("Rata-rata token sesudah preprocessing:", round(df["len_cleaned_text"].mean(), 2))
print("Dokumen jadi kosong setelah preprocessing:", int((df["len_cleaned_text"] == 0).sum()))

# contoh 5 data acak before-after
sample_cols = ["title", "full_text", "cleaned", "tokens", "tokens_clean", "tokens_stemmed", "cleaned_text"]
df[sample_cols].sample(min(5, len(df)), random_state=42)

Rata-rata token sebelum preprocessing: 40.27
Rata-rata token sesudah preprocessing: 37.08
Dokumen jadi kosong setelah preprocessing: 0


,title,full_text,cleaned,tokens,tokens_clean,tokens_stemmed,cleaned_text
95,Rancang bangun game 3D edukasi basic web devel...,Rancang bangun game 3D edukasi basic web devel...,Rancang bangun game 3D edukasi basic web devel...,"[rancang, bangun, game, 3d, edukasi, basic, we...","[rancang, bangun, game, 3d, edukasi, basic, we...","[rancang, bangun, game, 3d, edukasi, basic, we...",rancang bangun game 3d edukasi basic web devel...
15,A review of the application of machine learnin...,A review of the application of machine learnin...,A review of the application of machine learnin...,"[a, review, of, the, application, of, machine,...","[review, of, the, application, of, machine, le...","[review, of, the, application, of, machine, le...",review of the application of machine learning ...
30,Participation is not a design fix for machine ...,Participation is not a design fix for machine ...,Participation is not a design fix for machine ...,"[participation, is, not, a, design, fix, for, ...","[participation, is, not, design, fix, for, mac...","[participation, is, not, design, fix, for, mac...",participation is not design fix for machine le...
158,The role of mobile application acceptance in s...,The role of mobile application acceptance in s...,The role of mobile application acceptance in s...,"[the, role, of, mobile, application, acceptanc...","[the, role, of, mobile, application, acceptanc...","[the, role, of, mobile, application, acceptanc...",the role of mobile application acceptance in s...
128,A comprehensive review on cyber-attacks in pow...,A comprehensive review on cyber-attacks in pow...,A comprehensive review on cyber attacks in pow...,"[a, comprehensive, review, on, cyber, attacks,...","[comprehensive, review, on, cyber, attacks, in...","[comprehensive, review, on, cyber, attacks, in...",comprehensive review on cyber attacks in power...


In [20]:
# =========================================================
# SIMPAN HASIL PREPROCESSING
# =========================================================
base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
save_dir = os.path.join(base_dir, "data")
save_path = os.path.join(save_dir, "cleaned_papers.csv")

os.makedirs(save_dir, exist_ok=True)

save_df = df[
    ["id", "title", "abstract", "authors", "year", "source", "category", "pdf_url", "url", "scrape_status", "cleaned_text"]
].copy()

save_df.to_csv(save_path, index=False)

print(f"✅ Tersimpan: {save_path}")
print(f"📊 Total baris tersimpan: {len(save_df)}")
save_df.head(5)

✅ Tersimpan: d:\Tugas Akhir\paperci_artikel\backend\data\cleaned_papers.csv
📊 Total baris tersimpan: 200


,id,title,abstract,authors,year,source,category,pdf_url,url,scrape_status,cleaned_text
0,1,Machine learning for microbiologists,… how to evaluate a machine learning model and...,"F Asnicar, AM Thomas, A Passerini…",2024,Nature Reviews …,machine learning,https://pmc.ncbi.nlm.nih.gov/articles/PMC11980...,https://www.nature.com/articles/s41579-023-009...,pdf_failed,machine learning for microbiologists how to ev...
1,2,International conference on machine learning,"In this paper, we make the key delineation on ...","W Li, C Wang, G Cheng, Q Song",2023,Transactions on machine learning …,machine learning,https://par.nsf.gov/servlets/purl/10418406,https://par.nsf.gov/servlets/purl/10418406,pdf_failed,international conference on machine learning i...
2,3,What is machine learning?,… that one can employ in machine learning (ML)...,J Bell,2022,Machine learning and the city: applications in …,machine learning,None,https://onlinelibrary.wiley.com/doi/abs/10.100...,no_pdf,what is machine learning that one can employ i...
3,4,Amnesiac machine learning,… It gives EU residents the ability to request...,"L Graves, V Nagisetty, V Ganesh",2021,… of the AAAI conference on artificial …,machine learning,https://ojs.aaai.org/index.php/AAAI/article/do...,https://ojs.aaai.org/index.php/AAAI/article/vi...,pdf_downloaded,amnesiac machine learning it gives eu resident...
4,5,Designing nanotheranostics with machine learning,"… As a key branch of artificial intelligence, ...","L Rao, Y Yuan, X Shen, G Yu, X Chen",2024,Nature Nanotechnology,machine learning,https://www.researchgate.net/profile/Xi-Shen-1...,https://www.nature.com/articles/s41565-024-017...,pdf_failed,designing nanotheranostics with machine learni...


In [21]:
# =========================================================
# OPSIONAL: SIMPAN SAMPEL BUKTI TAHAPAN
# =========================================================
evidence_path = os.path.join(save_dir, "preprocessing_evidence_sample.csv")
evidence_cols = ["id", "title", "full_text", "cleaned", "tokens", "tokens_clean", "tokens_stemmed", "cleaned_text"]

df[evidence_cols].head(50).to_csv(evidence_path, index=False)
print(f"✅ Evidence sample tersimpan: {evidence_path}")

✅ Evidence sample tersimpan: d:\Tugas Akhir\paperci_artikel\backend\data\preprocessing_evidence_sample.csv
